# 10 — Ethical Perspectives on Geoprivacy

Map encryption is a technical tool, but deployment decisions are ethical ones.
This notebook translates the six core tensions from public health geoprivacy
practice into concrete scenarios using the 1854 Soho cholera dataset — the same
data encoded in NB06 and evaluated in NB08.

**Key argument:** "Protected" does not mean "harmless." A scheme that preserves
too much spatial structure may expose people; one that destroys too much may
mislead public health decisions. Both failures are ethical failures.

> Source framework: *Geoprivacy Ethics* (docs/Geoprivacy\ ethics.pdf), adapted
> from public health spatial data practice.

## The Six Core Tensions

Rather than a checklist of abstract principles, geoprivacy ethics is best
understood as a set of **contrasts** that practitioners must balance for each
deployment. None of the six tensions has a universal answer.

In [1]:
import plotly.graph_objects as go

tensions = [
    ["1. Individual privacy vs population benefit",
     "Protect people from exposure, stigma, surveillance, discrimination, legal risk",
     "Enable outbreak detection, resource targeting, environmental risk assessment",
     "What level of spatial detail is justified by the public health objective?"],
    ["2. Utility / fidelity vs disclosure risk",
     "Preserve true spatial structure; avoid fake clusters; keep local patterns interpretable",
     "Reduce the chance that a point, cluster, route, residence, or sensitive site can be inferred",
     "What spatial truths must remain visible, and which must be blurred?"],
    ["3. Equity and fairness vs efficiency",
     "Ask who bears the privacy risk; whether vulnerable groups face greater harm from disclosure",
     "Ask what enables fast, scalable, decision-relevant action",
     "Does the same protection level work equally across populations and geographies?"],
    ["4. Transparency and trust vs security through obscurity",
     "Explain what the protection does, what it does not do, and what risks remain",
     "Avoid operational details that would meaningfully weaken protection in deployment",
     "How do we describe the method honestly without creating false confidence?"],
    ["5. Stewardship / accountability vs technical capability",
     "Data minimization, role-based access, auditability, retention limits, governance",
     "Exact recovery, cross-dataset linkage, repeat-release analysis, granular dashboards",
     "What is the minimum spatial disclosure necessary for each role and use case?"],
    ["6. Emergency exception vs normal-rule restraint",
     "Urgent response may justify more granular data use",
     "Emergency logic should not quietly become the default for routine surveillance",
     "What temporary exceptions are justified, and what guardrails prevent permanent expansion?"],
]

fig = go.Figure(go.Table(
    columnwidth=[2.2, 2.8, 2.8, 2.8],
    header=dict(
        values=["Tension", "Protective side", "Operational side", "Framing question"],
        fill_color='#2c3e50', font=dict(color='white', size=11), align='left',
    ),
    cells=dict(
        values=list(zip(*tensions)),
        fill_color=[['#f0f4f8', '#e8edf2']*3]*4,
        align='left', font=dict(size=10),
        height=60,
    ),
))
fig.update_layout(title='Six core tensions in geoprivacy practice', height=500)
fig.show()


## Tension 2 in Practice: `SchemeParams` as an Ethical Dial

Every `SchemeParams` configuration encodes a position on the
**utility/fidelity vs disclosure risk** axis. The three parameters are not
merely technical choices — they are ethical ones:

| Parameter | Lower value → | Higher value → |
|-----------|---------------|----------------|
| `bin_size_m` | Finer grid; more sub-tile precision retained | Coarser grid; stronger spatial masking |
| `jitter_max_frac` | Display pins near tile centre; easier to cluster | Wider jitter; harder to fingerprint co-located records |
| `prp_rounds` | Faster; weaker shuffling at < 4 rounds | Stronger PRP; tile indices fully pseudorandom |

The Soho cholera data lets us measure this tradeoff concretely:
EDD (Expected Displacement Distance) captures the *display utility lost*;
DBSCAN cluster destruction captures the *disclosure risk eliminated*.

In [2]:
import secrets
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import DBSCAN

from map_encryption import (
    MapEncryption, SchemeParams,
    _project, _unproject,
    haversine_m, edd, mnnd, dbscan_f1,
)

MASTER_KEY = secrets.token_bytes(32)

deaths = pd.read_csv('data/cholera_deaths.csv')
true_coords = list(zip(deaths['LAT'].values, deaths['LON'].values))

# Sweep bin_size_m; hold jitter and rounds fixed.
bin_sizes = [50, 100, 250, 500, 1000]
results = []

for bs in bin_sizes:
    enc_i = MapEncryption(MASTER_KEY, SchemeParams(bin_size_m=bs))
    recs  = [enc_i.encode(lat, lon, tweak=MapEncryption.make_tweak(i))
             for i, (lat, lon) in enumerate(true_coords)]
    disp  = [enc_i.render_coordinates(r) for r in recs]

    tile_centers = [_unproject(r['qxp']*bs, r['qyp']*bs) for r in recs]
    edd_val  = edd(tile_centers, disp)
    mnnd_val = mnnd(disp)

    # DBSCAN on true coords (fixed reference)
    true_xy = np.array([_project(lat, lon) for lat, lon in true_coords])
    disp_xy = np.array([_project(lat, lon) for lat, lon in disp])
    db_true = DBSCAN(eps=300, min_samples=5).fit(true_xy)
    db_disp = DBSCAN(eps=300, min_samples=5).fit(disp_xy)
    n_true  = len(set(db_true.labels_)) - (1 if -1 in db_true.labels_ else 0)
    n_disp  = len(set(db_disp.labels_)) - (1 if -1 in db_disp.labels_ else 0)

    results.append({'bin_size_m': bs, 'EDD_m': edd_val,
                    'display_MNND_m': mnnd_val,
                    'true_clusters': n_true, 'display_clusters': n_disp})
    print(f'bin={bs:>4} m  EDD={edd_val:>7.1f} m  MNND={mnnd_val:>7.1f} m  '
          f'clusters: true={n_true}  display={n_disp}')

df = pd.DataFrame(results)


bin=  50 m  EDD=    4.4 m  MNND=1280599.2 m  clusters: true=1  display=0
bin= 100 m  EDD=    9.2 m  MNND=1294803.8 m  clusters: true=1  display=0
bin= 250 m  EDD=   22.1 m  MNND=1268605.4 m  clusters: true=1  display=0
bin= 500 m  EDD=   46.6 m  MNND=1302874.2 m  clusters: true=1  display=0
bin=1000 m  EDD=   86.5 m  MNND=1291870.9 m  clusters: true=1  display=0


In [3]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['EDD vs bin_size_m (display utility lost)',
                    'Display clusters vs bin_size_m (disclosure risk)'])

fig.add_trace(go.Scatter(x=df['bin_size_m'], y=df['EDD_m'],
    mode='lines+markers', marker=dict(size=8, color='steelblue'), name='EDD (m)'),
    row=1, col=1)

fig.add_trace(go.Scatter(x=df['bin_size_m'], y=df['display_clusters'],
    mode='lines+markers', marker=dict(size=8, color='tomato'), name='display clusters'),
    row=1, col=2)
fig.add_trace(go.Scatter(x=df['bin_size_m'],
    y=[df['true_clusters'].iloc[0]]*len(df), mode='lines',
    line=dict(dash='dash', color='gray'), name='true clusters (reference)'),
    row=1, col=2)

fig.update_xaxes(title_text='bin_size_m (m)', type='log')
fig.update_yaxes(title_text='EDD (metres)', row=1, col=1)
fig.update_yaxes(title_text='DBSCAN cluster count', row=1, col=2)
fig.update_layout(
    title='Utility–disclosure tradeoff: as bin_size_m grows, display utility falls and disclosure risk drops',
    template='plotly_white', height=420,
)
fig.show()

print('\nReading the chart:')
print(' Left panel  — larger bins move display pins further from tile centres (utility lost).')
print(' Right panel — larger bins destroy more cluster structure (disclosure risk reduced).')
print(' The default 250 m bin sits near the elbow of both curves.')



Reading the chart:
 Left panel  — larger bins move display pins further from tile centres (utility lost).
 Right panel — larger bins destroy more cluster structure (disclosure risk reduced).
 The default 250 m bin sits near the elbow of both curves.


## Ethical Principles Grounded in This Scheme

These eight principles are kept concrete by tying each to a specific scheme property:

In [4]:
principles = [
    ["Proportionality",
     "Intrusion or disclosure risk should be proportionate to the public health purpose.",
     "bin_size_m and jitter_max_frac set the disclosure floor; they should match the sensitivity of the use case, not default to maximum precision."],
    ["Data minimisation",
     "Collect, retain, and reveal no more spatial detail than necessary.",
     "Split storage (NB09): the display tier never receives ct_resid; the decode tier is invoked only when exact coordinates are operationally required."],
    ["Contextual integrity",
     "Information shared for care or surveillance should not be repurposed for unrelated monitoring or enforcement.",
     "The tweak binds each ciphertext to a stated purpose (extra= field). Records encoded for one context cannot be silently reused in another without re-encoding."],
    ["Justice / equity",
     "Evaluate whether privacy risks and analytic errors fall unevenly on vulnerable groups.",
     "Mercator distortion (NB07 Limitation 3) means a 250 m tile covers more real ground at higher latitudes — rural and northern communities get coarser spatial masking than urban ones."],
    ["Nonmaleficence",
     "Do not create avoidable harms: stigma, exposure, false inference, or misleading maps.",
     "Display jitter (NB05) prevents co-location fingerprinting that would identify households or clinics; PRP shuffle prevents neighbourhood-level inference from tile indices."],
    ["Beneficence",
     "Use spatial data in ways that produce genuine public health value.",
     "Cluster structure is preserved at the tile level (NB08 DBSCAN shows true clusters survive at bin=50 m) so outbreak detection remains viable even with encryption."],
    ["Accountability",
     "Make responsibilities, access tiers, assumptions, and residual risks explicit.",
     "Key privilege separation (NB05): jitter_key for display tier, prp_key + aead_key for decode tier, master key for admin — each tier's blast radius is bounded and auditable."],
    ["Respect for persons / communities",
     "Consider both individual and community-level consequences of spatial disclosure.",
     "The 1854 cholera data shows 171 of 250 deaths within 300 m of the Broadwick Street pump. Publishing exact addresses would expose households; the encrypted scheme shows cluster structure without revealing street-level locations."],
]

fig2 = go.Figure(go.Table(
    columnwidth=[1.8, 2.8, 3.4],
    header=dict(
        values=["Principle", "Definition", "How this scheme embodies it"],
        fill_color='#2c3e50', font=dict(color='white', size=11), align='left',
    ),
    cells=dict(
        values=list(zip(*principles)),
        fill_color=[['#f0f4f8', '#e8edf2']*4]*3,
        align='left', font=dict(size=10), height=70,
    ),
))
fig2.update_layout(title='Ethical principles and scheme properties', height=700)
fig2.show()


## Three Public Health Scenarios

The same scheme may be ethical in one scenario and inappropriate in another.
For each scenario the matrix shows which tensions dominate and what the
acceptable disclosure tier is.

In [5]:
scenarios = [
    {
        "name": "A — Infectious disease cluster response",
        "objective": "Detect and contain an outbreak (e.g., cholera, COVID-19); target resources to affected streets.",
        "perspectives": [
            ("Outbreak investigator", "Fine-grained spatial detail to identify point source"),
            ("Affected community member", "Protection from identification, stigma, and quarantine enforcement"),
            ("Public communicator", "A map that informs without exposing neighbourhoods or households"),
        ],
        "tensions": "Privacy vs public benefit · Emergency exception vs restraint · Transparency vs trust",
        "acceptable_tier": "Decode tier with audit log; display tier for public-facing maps",
        "scheme_fit": "Good fit. 250 m bins preserve street-block clusters. Tweak binds records to investigation context; re-identification risk is low.",
    },
    {
        "name": "B — Substance use / overdose / mental health services",
        "objective": "Place harm-reduction services at overdose hotspots; evaluate coverage gaps.",
        "perspectives": [
            ("Service planner", "Hotspot accuracy for service placement"),
            ("Affected community", "Stigma, policing, and reputational harm from disclosure"),
            ("Ethics reviewer", "Indirect identification through small-area clustering"),
        ],
        "tensions": "Utility vs disclosure risk · Equity vs efficiency · Stewardship vs capability",
        "acceptable_tier": "Display tier only for public reports; decode tier restricted to licensed analysts under DUA",
        "scheme_fit": "Use with caution. Small, stigmatised populations are more re-identifiable even at 250 m. Consider bin_size_m ≥ 500 m and suppressing cells with < 5 records.",
    },
    {
        "name": "C — Environmental exposure or access-to-care mapping",
        "objective": "Map pollution burden or healthcare deserts at neighbourhood level to support equity analysis.",
        "perspectives": [
            ("Analyst", "Neighbourhood-level pattern preservation; structural inequities visible"),
            ("Resident", "Representation of inequity without individual exposure"),
            ("Privacy officer", "Low re-identification risk for rare or severe cases"),
        ],
        "tensions": "Fidelity vs disclosure risk · Equity vs efficiency · Transparency vs obscurity",
        "acceptable_tier": "Display tier with aggregated counts; decode tier only for formal research under IRB",
        "scheme_fit": "Good fit for aggregate display. Individual records should remain in ct_resid vault (NB09 split storage); publish only tile-level counts.",
    },
]

for sc in scenarios:
    fig_s = go.Figure(go.Table(
        columnwidth=[1.5, 3.5],
        header=dict(
            values=[f'<b>{sc["name"]}</b>', ''],
            fill_color='#34495e', font=dict(color='white', size=12), align='left',
        ),
        cells=dict(
            values=[
                ["Objective", "Perspective contrasts", "Key tensions",
                 "Acceptable disclosure tier", "Scheme fit"],
                [sc["objective"],
                 "\n".join(f'• {role}: {view}' for role, view in sc["perspectives"]),
                 sc["tensions"],
                 sc["acceptable_tier"],
                 sc["scheme_fit"]],
            ],
            fill_color=[['#ecf0f1'], ['#ffffff']],
            align='left', font=dict(size=10), height=60,
        ),
    ))
    fig_s.update_layout(height=380, margin=dict(t=20, b=10))
    fig_s.show()


## What to Avoid

1. **"Protected" ≠ "harmless."** The display tier still reveals approximate location. A jittered pin inside a 250 m tile centred on a rural clinic is sufficient to identify it.

2. **"Encrypted" ≠ "anonymous."** The scheme provides semantic security for the exact coordinates; it does not provide k-anonymity or differential privacy. An attacker who can correlate tile indices across multiple releases can narrow location estimates.

3. **Preserving morphology is itself a disclosure decision.** Showing that deaths cluster around a pump (NB08) is intentional. That same cluster structure could expose a household, clinic, or stigmatised service in a different context.

4. **Inaccurate protection is also unethical.** A scheme that invents spatial patterns where none exist misleads public health decisions. Both over-protection and under-protection can cause harm.

5. **Emergency exceptions creep.** A bin_size_m reduction granted for outbreak response should have a defined expiry. Operational systems tend to retain the more granular setting after the emergency ends.

## Summary

| Technical choice | Ethical dimension |
|-----------------|-------------------|
| `bin_size_m` | Proportionality — how much spatial detail is justified by the use case? |
| `jitter_max_frac` | Nonmaleficence — does display jitter prevent co-location fingerprinting? |
| `prp_rounds` | Accountability — is the shuffle strong enough to bound blast radius? |
| Split storage (NB09) | Data minimisation — display tier never touches `ct_resid` |
| Key privilege separation (NB05) | Stewardship — each tier holds only the keys it operationally needs |
| Tweak / record binding | Contextual integrity — ciphertexts are bound to the purpose of encoding |
| Version field | Accountability — records carry their creation-time scheme version for auditability |

The six tensions do not resolve into a single answer. They resolve into a
**decision process** — one that should involve the data subjects, public health
practitioners, ethics reviewers, and technologists together.

## References

- **Geoprivacy Ethics** (2024). *Ethical tensions in public health geoprivacy.* Internal framework document; see `docs/Geoprivacy ethics.pdf`.
- **Snow, J.** (1855). *On the Mode of Communication of Cholera* (2nd ed.). Churchill, London.
- **Brody, H., Rip, M.R., Vinten-Johansen, P., Paneth, N., & Rachman, S.** (2000). Map-making and myth-making in Broad Street: the London cholera epidemic, 1854. *The Lancet, 356*(9223), 64–68.
- **Lin, J.** (2023). Geo-indistinguishable masking: enhancing privacy protection in spatial point mapping. See `docs/Geo-indistinguishablemaskingenhancingprivacyprotectioninspatialpointmapping.pdf`.
- **Dwork, C.** (2006). Differential privacy. *Proceedings of ICALP 2006*, LNCS 4052, 1–12.
- **Sweeney, L.** (2002). k-anonymity: a model for protecting privacy. *International Journal of Uncertainty, Fuzziness and Knowledge-Based Systems, 10*(5), 557–570.
- **Luby, M., & Rackoff, C.** (1988). How to construct pseudorandom permutations from pseudorandom functions. *SIAM Journal on Computing, 17*(2), 373–386.
- **Nir, Y., & Langley, A.** (2018). ChaCha20 and Poly1305 for IETF Protocols. RFC 8439. IETF.